# Video1 reconstruction (GPU)

Runs `reconstruct_video()`'s full-video, ICP-refined reconstruction against
the user's real endoscope video (`Video1.avi`, uploaded as the private
`endoslam-video1-sample` dataset) with GPU on.

**Why GPU here when Phase 4/5 ran GPU-off**: local CPU profiling found the
per-frame cost is dominated by DarkIR-lite/Mini-3D-Recon's forward passes
(~2.6s + ~0.5s per pipeline step) with ICP a smaller ~0.17s slice that stays
CPU-bound regardless of hardware (Open3D's registration_icp doesn't use
GPU). A P100/T4 should meaningfully cut the forward-pass cost even though
it won't speed up ICP itself. The sliding-window's redundant DarkIR-lite
recomputation -- every frame was
being re-enhanced up to `context_window`=8 times, once per overlapping
window it fell in -- is now fixed separately (`enhance_all_frames()`
enhances each frame exactly once and caches it), so DarkIR-lite is no
longer the dominant per-frame cost it used to be.

Same Pascal/P100 torch-kernel fix as the Phase 2/3 training notebooks --
Kaggle's preinstalled torch (2.10.0) has zero sm_60 kernels, so the
default GPU is unusable without this reinstall.

## 0. Setup: clone repo + DarkIR upstream, reinstall Pascal-compatible torch, install deps

In [ ]:
REPO_URL = "https://github.com/ritiksharma3/endoslam.git"
DARKIR_URL = "https://github.com/cidautai/DarkIR.git"

!git clone $REPO_URL repo
!git clone $DARKIR_URL repo/DarkIR_upstream
%cd repo

# Same deliberate exception to "never touch preinstalled torch on Kaggle" as
# the Phase 2/3 training notebooks -- the preinstalled build has zero
# Pascal (sm_60) kernels, so every real CUDA op fails on the P100 Kaggle
# assigns here regardless. NOTE: --extra-index-url, not --index-url --
# --index-url replaces PyPI entirely and breaks transitive deps like
# nvidia-cudnn-cu12.
!pip install -q torch==2.5.1 torchvision==0.20.1 --extra-index-url https://download.pytorch.org/whl/cu121

!pip install -q -r environment/requirements.txt

## 1. GPU check + resolve the video file and both checkpoints

In [ ]:
import os
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

gpu_actually_usable = False
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    try:
        torch.zeros(1, device="cuda") + torch.zeros(1, device="cuda")
        gpu_actually_usable = True
        print("GPU FIX CONFIRMED: real CUDA op succeeded")
    except RuntimeError as e:
        print(f"GPU still not usable after torch reinstall: {e}")


def find_file(base, extensions):
    for root, _dirs, files in os.walk(base):
        for f in files:
            if os.path.splitext(f)[1].lower() in extensions:
                return os.path.join(root, f)
    return None


# Kaggle mounts dataset_sources under a nested owner-scoped path
# (/kaggle/input/datasets/<owner>/<slug>/...), not the flat
# /kaggle/input/<slug>/... path older docs/examples assume -- confirmed by
# walking /kaggle/input on a failed run. Search from /kaggle/input itself
# (same pattern find_phase_checkpoint below already uses) so this doesn't
# depend on Kaggle's exact mount layout.
VIDEO_PATH = find_file("/kaggle/input", {".avi", ".mp4"})
if not VIDEO_PATH:
    print("VIDEO_PATH not found. Contents of /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count(os.sep) - "/kaggle/input".count(os.sep)
        if depth > 2:
            dirs[:] = []
            continue
        print(f"  {root}: dirs={dirs} files={files}")
assert VIDEO_PATH, "could not find the uploaded video anywhere under /kaggle/input"
print("VIDEO_PATH:", VIDEO_PATH)


def find_phase_checkpoint(name_fragment: str, base="/kaggle/input"):
    # Two kernel_sources are mounted (Phase 2 and Phase 3 training outputs),
    # each with files named epoch_*.pt -- filter by a name fragment unique
    # to each kernel's slug to disambiguate, same trick phase5_evaluation
    # used for the same reason.
    candidates = [os.path.join(r, f) for r, _, fs in os.walk(base) for f in fs
                  if f.startswith("epoch_") and f.endswith(".pt") and name_fragment in r.lower()]
    if not candidates:
        return None
    return max(candidates, key=lambda p: int(os.path.basename(p).split("_")[1].split(".")[0]))


DARKIR_CHECKPOINT_PATH = find_phase_checkpoint("darkir")
assert DARKIR_CHECKPOINT_PATH, "could not find a Phase 2 DarkIR-lite checkpoint under /kaggle/input"
print("DARKIR_CHECKPOINT_PATH:", DARKIR_CHECKPOINT_PATH)

MINI_RECON_CHECKPOINT_PATH = find_phase_checkpoint("mini3drecon")
assert MINI_RECON_CHECKPOINT_PATH, "could not find a Phase 3 Mini-3D-Recon checkpoint under /kaggle/input"
print("MINI_RECON_CHECKPOINT_PATH:", MINI_RECON_CHECKPOINT_PATH)

## 2. Load config and both trained models

In [ ]:
import yaml

from src.common.device import select_device
from src.darkir_lite.model import build_darkir_lite
from src.reconstruction.model import MiniReconModel

with open("configs/config.yaml") as f:
    config = yaml.safe_load(f)

device = select_device()
print("device:", device)

darkir_model = build_darkir_lite(pretrained=False).to(device)
darkir_checkpoint = torch.load(DARKIR_CHECKPOINT_PATH, map_location=device, weights_only=False)
darkir_model.load_state_dict(darkir_checkpoint["model_state_dict"])
print(f"loaded DarkIR-lite: epoch={darkir_checkpoint['epoch']}, val_psnr={darkir_checkpoint.get('val_psnr')}")

mini_recon_model = MiniReconModel(
    pretrained=False, depth_head_channels=config["reconstruction"]["depth_head_channels"]
).to(device)
mini_recon_checkpoint = torch.load(MINI_RECON_CHECKPOINT_PATH, map_location=device, weights_only=False)
mini_recon_model.load_state_dict(mini_recon_checkpoint["model_state_dict"])
print(f"loaded Mini-3D-Recon: epoch={mini_recon_checkpoint['epoch']}, "
      f"val_depth_absrel={mini_recon_checkpoint.get('val_depth_absrel')}")

## 3. Load video frames + reconstruct

`FRAME_STRIDE = 1` -- back to full density. The first full-video run
used `FRAME_STRIDE = 125` (~1 frame/5s) for speed, but the resulting
point cloud looked like stacked layers/a blob instead of a trajectory
through the stomach. Root cause: Mini-3D-Recon's pose head was trained
exclusively on consecutive UnityCam frames (~33ms apart) and never
learned to scale its output to larger real motion -- at stride 125, a
125x larger real time-gap only grew predicted translation ~2x (and
predicted rotation didn't grow at all), so the pose chain barely moved
the camera between frames while a single frame's own point cloud
already spans ~100-200x more space than that tiny predicted step --
every new frame lands almost entirely inside the space prior frames
already occupy. ICP can't fix this (it's a bounded local refinement
with no independent sense of absolute scale, built to correct drift,
not inject missing motion) -- widening its search radius, as the
stride-125 run did, only pulled frames tighter together and made it
worse. Running at full density keeps pose prediction in-distribution,
so it should produce a real trajectory instead of a blob.

`ICP_MAX_CORR_DIST` is back to `reconstruct_video()`'s own default (10x
`fusion.voxel_downsample`) -- that default is tuned for the small,
consecutive-frame motion this stride now matches; the earlier widening
was specifically compensating for stride 125's much larger gaps and
would be counterproductive here.

Checkpointing is essential at this density (~3000 frames, not ~24) --
`CHECKPOINT_EVERY` fused frames, so a Kaggle session interruption loses
at most that many frames of fusion-loop progress instead of the whole
run. The forward-pass phase (DarkIR-lite + Mini-3D-Recon) now enhances
each frame exactly once instead of up to 8x (see `enhance_all_frames()`
in `src/inference/reconstruct_video.py`), so this full-density run
should be well under the original ~7h baseline -- expect it to land
somewhere under an hour on GPU, though this is the first run at full
density with both fixes in place, so treat that as an estimate, not a
guarantee.

In [ ]:
import time

from src.inference.reconstruct_video import load_video_frames, reconstruct_video

FRAME_STRIDE = 1  # full density -- see markdown above for why (stacking/domain-gap root cause)
CHECKPOINT_PATH = "/kaggle/working/checkpoints/video1_reconstruction.pt"
CHECKPOINT_EVERY = 200  # matches config.yaml's reconstruction.checkpoint_every_steps default
RESUME = False  # set True and re-run this cell to continue from CHECKPOINT_PATH after an interruption

image_size = tuple(config["data"]["image_size"])
t0 = time.time()
frames = load_video_frames(VIDEO_PATH, image_size, frame_stride=FRAME_STRIDE)
print(f"loaded {frames.shape[0]} frames from {VIDEO_PATH} ({time.time() - t0:.1f}s)")

t0 = time.time()
pcd = reconstruct_video(
    frames, darkir_model, mini_recon_model, config, device,
    checkpoint_path=CHECKPOINT_PATH,
    checkpoint_every=CHECKPOINT_EVERY,
    resume=RESUME,
)
elapsed = time.time() - t0
print(f"reconstructed point cloud: {len(pcd.points)} points in {elapsed:.1f}s "
      f"({elapsed / frames.shape[0]:.3f}s/frame)")

## 4. Save .ply + preview PNG

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d

os.makedirs("/kaggle/working/output", exist_ok=True)
PLY_PATH = "/kaggle/working/output/video1_fullvideo_icp_gpu.ply"
PREVIEW_PATH = "/kaggle/working/output/video1_fullvideo_icp_gpu_preview.png"

o3d.io.write_point_cloud(PLY_PATH, pcd)
print("saved:", PLY_PATH)

pts = np.asarray(pcd.points)
cols = np.asarray(pcd.colors)
max_points = 30000
if len(pts) > max_points:
    idx = np.random.choice(len(pts), max_points, replace=False)
    pts, cols = pts[idx], cols[idx]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (i, j, label) in zip(axes, [(0, 2, "top (X-Z)"), (1, 2, "side (Y-Z)"), (0, 1, "front (X-Y)")]):
    ax.scatter(pts[:, i], pts[:, j], c=cols, s=0.5)
    ax.set_title(label)
    ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(PREVIEW_PATH, dpi=100)
print("saved:", PREVIEW_PATH)

## Done

Download `output/video1_fullvideo_icp_gpu.ply` and the preview PNG. Compare
the printed `s/frame` figure against the local CPU baseline (~3.0s/frame)
to see the actual GPU speedup -- ICP's ~0.17s/frame CPU-bound floor means
the theoretical best case is still bounded, not proportional to raw GPU
throughput.